Aqui va el entrenamiento del Modelo

In [1]:
# Importacion de librerias
import fitz
import logging
import os
import re
import string
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaLLM
import langchain
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document as LCDocument
from langchain_core.embeddings import Embeddings

In [2]:
# Verificar la orquestacion con LangChain
print(f"Versión de LangChain: {langchain.__version__}")
try:
    llm = OllamaLLM(model="llama3")
    print("Modelo LLaMA 3 listo para orquestar.")
except Exception as e:
    print("LangChain instalado, pero el servicio de LLaMA no está activo.")

Versión de LangChain: 1.2.10
Modelo LLaMA 3 listo para orquestar.


In [3]:
# Creacion de logger
BASE_DIR = os.path.dirname(os.path.abspath('/Users/marcela/Library/CloudStorage/OneDrive-Personal/Clases/ML1 - Proyecto Final/Backend_AprendizajeDeMaquinaI/entrenamiento/logs'))
LOG_FILE_PATH = os.path.join(BASE_DIR, "extraccion.log")
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE_PATH), 
        logging.StreamHandler()             
    ]
)
logger = logging.getLogger(__name__)

In [4]:
def txt_y_metadatos(pdf_path):
    doc = fitz.open(pdf_path)
    metadata = doc.metadata
    full_text = []
    headers = []
    
    for page in doc:
        full_text.append(page.get_text())
        blocks = page.get_text("dict")["blocks"]
        for b in blocks:
            if b['type'] == 0: 
                for line in b["lines"]:
                    for span in line["spans"]:
                        if span.get("color") == 19072 and span["text"].isupper():
                            headers.append(span["text"].strip())
    

    texto_completo = "\n".join(full_text)
    metadata["headers_extraidos"] = list(dict.fromkeys(headers)) 
    
    return texto_completo, metadata

In [5]:
# Limpiar el texto
def limpiar_texto(texto):
    texto = texto.lower()
    texto = texto.translate(str.maketrans('','',string.punctuation))
    texto= texto.strip()
    return texto

In [6]:
# Hacer el chunking del texto
def chunk_text(text):
    splitter = RecursiveCharacterTextSplitter(
        # Al hacer el chunck_size y el chunck_overlap pequeños, nos aseguramos que el modelo sea mas especifico
        chunk_size=400,
        chunk_overlap=50
    )
    return splitter.split_text(text)

In [7]:
# Generar los embeddings

# Pre-requisitos para BioSentVec
import sent2vec

model_path = './BioSentVec_model/BioSentVec_PubMed_MIMICIII-bigram_d700.bin'
model = sent2vec.Sent2vecModel()
try:
    model.load_model(model_path)
except Exception as e:
    print(e)
print('Modelo guardado exitosamente')



Modelo guardado exitosamente


In [8]:
# Ocupamos envolver el modelo en un wrapper para que sea compatible con LangChain
# Clase traductora de BioSentVec a formato compatible con LangChain
class BioSentVecWrapper(Embeddings): 
    def __init__(self, model):
        self.model = model

    def embed_documents(self, texts):
        return self.model.embed_sentences(texts).tolist()

    def embed_query(self, text):
        return self.model.embed_sentences([text])[0].tolist()

bio_wrapper = BioSentVecWrapper(model)

In [ ]:
def guardar_en_faiss(chunks, vectores, metadatos_archivo, faiss_path):
    try:
        documentos = []
        for i, chunk in enumerate(chunks):
            doc = LCDocument(
                page_content=chunk,
                metadata={
                    "source": metadatos_archivo.get("file", "PDF_Renal"),  
                    "headers": metadatos_archivo.get("headers_extraidos", [])[:5]
                }
            )
            documentos.append(doc)

        vector_store = FAISS.from_documents(
            documents=documentos,
            embedding=bio_wrapper
        )
        
        os.makedirs(os.path.dirname(faiss_path), exist_ok=True)
        vector_store.save_local(faiss_path)
        return vector_store
    except Exception as e:
        print(f"❌ Error en FAISS: {e}")
        return None


In [10]:
def generar_embeddings(chunks):
    try:
        embeddings = model.embed_sentences(chunks)
        return embeddings 
    except Exception as e:
        logger.error(f"Error generando embeddings: {e}")
        return None

In [11]:
# ============================================================
# 🧪 CELDA DE PRUEBA: Pipeline completo + validación FAISS
# ============================================================

import os
import numpy as np

# ── 1. Parámetros ────────────────────────────────────────────
PDF_PATH    = "./Documentos/seccion_renal.pdf"          # <-- cambia aquí
FAISS_PATH  = "./FAISS/faiss_index_renal"
QUERY_TEST = "procesos renales"  # en inglés   # <-- cambia a algo que esté en tu PDF

# ── 2. Extraer texto y metadatos ─────────────────────────────
print("📄 Extrayendo texto y metadatos...")
texto, metadatos = txt_y_metadatos(PDF_PATH)

print(f"  ✅ Texto extraído: {len(texto):,} caracteres")
print(f"  📌 Headers encontrados ({len(metadatos['headers_extraidos'])}):")
for h in metadatos["headers_extraidos"][:5]:
    print(f"     • {h}")

# ── 3. Chunking ───────────────────────────────────────────────
print("\n✂️  Haciendo chunking...")
chunks = chunk_text(texto)
print(f"  ✅ Total de chunks: {len(chunks)}")
print(f"  📏 Tamaño promedio: {np.mean([len(c) for c in chunks]):.0f} chars")
print(f"  🔍 Primer chunk:\n     {chunks[0][:200]}...")

# ── 4. Generar embeddings ─────────────────────────────────────
print("\n🧬 Generando embeddings...")
vectores = generar_embeddings(chunks)

if vectores is None:
    print("  ❌ Error generando embeddings. Abortando.")
else:
    vectores_np = np.array(vectores)
    print(f"  ✅ Shape de vectores: {vectores_np.shape}")
    print(f"  📐 Norma del primer vector: {np.linalg.norm(vectores_np[0]):.4f}")

    # ── 5. Guardar en FAISS ───────────────────────────────────
    print("\n💾 Guardando en FAISS...")
    vector_store = guardar_en_faiss(chunks, vectores, metadatos, FAISS_PATH)

    if vector_store is None:
        print("  ❌ Error guardando en FAISS.")
    else:
        print(f"  ✅ FAISS guardado en: {FAISS_PATH}")

        # ── 6. Validación: buscar por similitud ───────────────
    from langchain_community.retrievers import BM25Retriever
    from langchain_classic.retrievers import EnsembleRetriever


    documentos_bm25 = [
        LCDocument(
            page_content=chunk,
            metadata={
                "source": metadatos.get("file", "PDF_Renal"),
                "headers": metadatos.get("headers_extraidos", [])[:5]
            }
        )
        for chunk in chunks
    ]

    bm25_retriever = BM25Retriever.from_documents(documentos_bm25)
    bm25_retriever.k = 3

    faiss_retriever = vector_store.as_retriever(search_kwargs={"k": 3})

    ensemble = EnsembleRetriever(
        retrievers=[bm25_retriever, faiss_retriever],
        weights=[0.5, 0.5]
    )

    # ── 7. Validación: buscar con ensemble ────────────────
    print(f"\n🔎 Buscando: '{QUERY_TEST}'")
    resultados = ensemble.invoke(QUERY_TEST)

    print(f"\n{'─'*60}")
    print(f"  Top {len(resultados)} resultados:")
    print(f"{'─'*60}")

    for i, doc in enumerate(resultados, 1):
        print(f"\n  [{i}]")
        print(f"  📌 Source : {doc.metadata.get('source', 'N/A')}")
        print(f"  🏷️  Headers: {doc.metadata.get('headers', [])}")
        print(f"  📝 Chunk  :\n     {doc.page_content[:300]}...")


        # ── 8. Resumen final ──────────────────────────────────
        print(f"\n\n{'='*60}")
        print("📊 RESUMEN DEL PIPELINE")
        print(f"{'='*60}")
        print(f"  Chunks generados   : {len(chunks)}")
        print(f"  Dimensión vectores : {vectores_np.shape[1]}")
        print(f"  FAISS guardado     : ✅")
        print(f"  FAISS recargado    : ✅")
        print(f"  Búsqueda funcional : ✅")
        print(f"{'='*60}")

📄 Extrayendo texto y metadatos...
  ✅ Texto extraído: 301,971 caracteres
  📌 Headers encontrados (74):
     • ASCITIS
     • COMPLICACIONES
     • ALTERACIONES EN LA FUNCIÓN RENAL Y DE LAS VÍAS URINARIAS
     • DISURIA
     • HIPERAZOEMIA

✂️  Haciendo chunking...
  ✅ Total de chunks: 850
  📏 Tamaño promedio: 359 chars
  🔍 Primer chunk:
     Manifestaciones cardinales y presentación de enfermedades
288
PARTE 2
Conviene practicar otros estudios sólo en circunstancias clínicas espe-
cífi cas. Si se sospecha peritonitis que es consecuencia d...

🧬 Generando embeddings...
  ✅ Shape de vectores: (850, 700)
  📐 Norma del primer vector: 7.6949

💾 Guardando en FAISS...
  ✅ FAISS guardado en: ./FAISS/faiss_index_renal

🔎 Buscando: 'procesos renales'

────────────────────────────────────────────────────────────
  Top 6 resultados:
────────────────────────────────────────────────────────────

  [1]
  📌 Source : PDF_Renal
  🏷️  Headers: ['ASCITIS', 'COMPLICACIONES', 'ALTERACIONES EN LA FUNCIÓN REN

In [ ]:
# Script para entender el formato del documento PDF
doc = fitz.open("./Documentos/seccion_renal.pdf")
for i in range(1): # Solo la primera página
    for block in doc[i].get_text("dict")["blocks"]:
        if "lines" in block:
            for line in block["lines"]:
                for span in line["spans"]:
                    if len(span['text'].strip()) > 3:
                        print(f"Texto: {span['text']} | Tamaño: {span['size']} | Color: {span['color']}")

Texto: Manifestaciones cardinales y presentación de enfermedades | Tamaño: 11.5 | Color: 2301728
Texto: PARTE 2 | Tamaño: 13.5 | Color: 16777215
Texto: Conviene practicar otros estudios sólo en circunstancias clínicas espe- | Tamaño: 8.909544944763184 | Color: 2301728
Texto: cífi cas. Si se sospecha peritonitis que es consecuencia de la perforación de  | Tamaño: 8.909544944763184 | Color: 2301728
Texto: una víscera habrá que medir las concentraciones de glucosa y lactato des- | Tamaño: 8.909544944763184 | Color: 2301728
Texto: hidrogenasa en el líquido de ascitis (LDH,  | Tamaño: 8.909544944763184 | Color: 2301728
Texto: lactate dehydrogenase | Tamaño: 8.909544944763184 | Color: 2301728
Texto: ). A dife- | Tamaño: 8.909544944763184 | Color: 2301728
Texto: rencia de la peritonitis bacteriana “espontánea” (SBP, | Tamaño: 8.909544944763184 | Color: 2301728
Texto:  spontaneus bacterial  | Tamaño: 8.909544944763184 | Color: 2301728
Texto: peritonitis | Tamaño: 8.909544944763184 | Color: 230

In [13]:

# En lugar de langchain.chains, usa:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate

retriever = vector_store.as_retriever(search_kwargs={"k": 3})
llm = OllamaLLM(model="llama3.2",temperature=0.2)

# Definir el prompt
#system_prompt = (
    #"Eres un medico experto y con mucho conocimiento y experiencia en la fisiologia renal, en especial las alteraciones "
    #"en la funcion renal y las vias urinarias. Tambien eres un maestro con mucha experiencia en la enseñanza de temas médicos."
   # "Tu trabajo es explicar de una forma profunda y didáctica el contenido del contexto. No asumas que las personas conocen los "
   # "terminos del contexto. Explica todos los terminos que usas. Si no sabes una respuesta o no se encuentra dentro de los temas de tu "
    #"base de datos, responde con 'No tengo información sobre ese tema.' Si deciden hacerte una pregunta no relacionada con terminos medicos, cordialmente"
   # "explica que no puedes proveer informacion sobre ese tema y que los impulsas a preguntar en otro lado. El maestro que eres es gentil y paceinte."
   # "Nunca te sobreexaltas ni eres grosero o frio con las demas personas. Siempre responde de manera amigable, respetuosa, cordial y amable. Se carismatico"
   # "pero profesional al mismo tiempo. : {context}"
#)
system_prompt = (
    "Eres el Dr. Nefros, un médico especialista en nefrología y urología con amplia experiencia clínica y docente. "
    "Tu conocimiento está basado exclusivamente en la Sección 7 del Harrison: Principios de Medicina Interna, "
    "que cubre la función renal y las vías urinarias.\n\n"

    "## TU ROL COMO EDUCADOR\n"
    "- Explica los temas de forma progresiva: comienza con conceptos básicos antes de profundizar.\n"
    "- Define SIEMPRE los términos médicos o técnicos la primera vez que los uses. Ejemplo: 'la creatinina (una sustancia de desecho que filtra el riñón)...'\n"
    "- Usa analogías cotidianas para facilitar la comprensión cuando sea posible.\n"
    "- Estructura tus respuestas con claridad: usa párrafos cortos, y cuando sea útil, listas o pasos numerados.\n\n"

    "## MANEJO DEL CONTEXTO\n"
    "- Basa tus respuestas ÚNICAMENTE en el contexto proporcionado a continuación.\n"
    "- Si la pregunta es médica pero no está cubierta en el contexto, responde: "
    "'Esa pregunta es interesante, pero no tengo información sobre ese tema en mi base de conocimientos actual. "
    "Te recomiendo consultar directamente el Harrison o un especialista.'\n"
    "- No inventes información ni extrapoles más allá de lo que indica el contexto.\n\n"

    "## TONO Y COMUNICACIÓN\n"
    "- Sé siempre amable, paciente, empático y motivador. Nunca uses un tono frío o condescendiente.\n"
    "- Si alguien comete un error conceptual, corrígelo con gentileza y sin hacerle sentir mal.\n"
    "- Si te hacen preguntas no relacionadas con medicina renal o vías urinarias, responde con amabilidad:\n"
    "  'Me especializo en temas de función renal y vías urinarias. Para esta pregunta, te sugiero "
    "consultar otra fuente o especialista. ¿Hay algo relacionado con los riñones en lo que pueda ayudarte?'\n\n"

    "## CONTEXTO DE REFERENCIA\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])


question_answer_chain = create_stuff_documents_chain(
    llm, 
    prompt, 

)

rag_chain = create_retrieval_chain(retriever, question_answer_chain)

pregunta = "¿Qué procesos ocurren en el riñón según el texto?"
print(f"🤔 Preguntando a LLaMA 3: {pregunta}")

try:
    response = rag_chain.invoke({"input": pregunta})
    print("\n🤖 RESPUESTA DE LLaMA 3:")
    print(response["answer"])
except Exception as e:
    print(f"❌ Error en la cadena: {e}")


🤔 Preguntando a LLaMA 3: ¿Qué procesos ocurren en el riñón según el texto?


2026-03-01 23:30:59,595 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"



🤖 RESPUESTA DE LLaMA 3:
¡Hola! Me alegra ayudarte a entender cómo funciona nuestro increíble órgano, el riñón.

Según el texto, el riñón es responsable de eliminar los desechos del cuerpo, como la urea y la creatinina. Estos procesos ocurren en varias etapas:

1. **Filtración glomerular**: En el interior del espacio urinario, llamado glomérulo, se filtra la urea y la creatinina desde el torrente sanguíneo hacia el espacio urinario.
2. **Depuración renal**: La urea y la creatinina se excretan en la orina a través de los túbulos del riñón. Sin embargo, la urea se absorbe y secreta en el túbulo, mientras que la creatinina no se absorbe ni se secreta.
3. **Cálculo de GFR**: Para medir la función renal, se utiliza una sustancia llamada iotalamato (o inulina), que se filtra en el glomérulo y se excreta en la orina. La velocidad con que aparece este isótopo en la orina durante varias horas se utiliza para calcular la función renal, conocida como GFR (Glomerular Filtración Rate).

Es importan